<a href="https://colab.research.google.com/github/RidwanRidu/Bengali-English-CodeMixed-Sentiment-Analysis/blob/main/Sentiment_Analysis_of_Bengali_Social_Media_Posts_using_BERT_based_Transformer_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

url = "Train.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

Shape: (12575, 2)

Columns: ['Data', 'Label']

First 5 rows:
                                                Data  Label
0  মুগ্ধ হয়ে গেলাম মামু. আর তোমায় কি কমু. বলো তোম...      1
1  এই কুত্তার বাচ্চাদের জন্য দেশটা আজ এমন অবস্তায়...      2
2                          ভাই আপনার কথাই যাদু রয়েছে      1
3                        উওরটা আমার অনেক ভাল লেগেছে       1
4  আমার নিজের গাড়ী নিয়ে কি সাজেক যেতে পারবো না ?...      0


In [21]:
label_col = 'Data'   # আপনার কলামের নাম অনুযায়ী বদলান
print(df[label_col].value_counts())

Data
কুত্তারবাচ্ছা , তোর জন্মই ত পতিতালয়ে । তা না হলে অন্যর মাকে গালি দিতি না । তোর মা বোন বেশ্যা । বেশ্যাগিরি করে তোদের জীবন চলে । খাংকি মাগির পোলা     3
খুবই ভালো জিনিস নিয়ে আপনারা আগ্রহ দেখাচ্ছেন । গান বাজনা হারাম । ছেলেটাকে গায়ক না বানিয়ে পড়াশোনায় সাহায্য করুন , সে ডাক্তার হতে চায়                 3
ভাইয়া অনেক সুন্দর হইছে । ভাইয়া কয়েকদিন ধরে আপনি ♥ দেন না , ফুরিয়ে গেছে নাকি                                                                         3
এই অবস্থা শুধু বুয়েটে না প্রত্যেকটা পাবলিক বিশ্ববিদ্যালয় , প্রত্যেকটা সরকারি কলেজের রেগিং নামে শারীরিক ও মানসিক অত্যাচার করা হয়                  3
আমিও শ্বপ্ন তে কাজ করি , ভালোই খারাপ না                                                                                                             3
                                                                                                                                                   ..
দাদা দারুন লাগলো । গুলশান এ কোন রেস্টরেন্টে ভালো ইলিশ পাওয়া যায়                              

In [22]:
# কলাম রিনেম
df = df.rename(columns={'Data': 'text', 'Label': 'label'})

# মিসিং ভ্যালু চেক
print(df.isnull().sum())
df = df.dropna(subset=['text', 'label'])

text     0
label    0
dtype: int64


In [23]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[['text', 'label']])
print(dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 12575
})


In [24]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset['train']
val_dataset = dataset['test']

In [25]:
# ৫. Tokenizer ও Model লোড
model_name = "sagorsarker/bangla-bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
# ৬. টোকেনাইজেশন
def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset = val_dataset.map(tokenize_fn, batched=True)

# লেবেল কলামকে 'labels' নাম দিন (Trainer-এর চাহিদা)
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")

# PyTorch ফরম্যাটে সেট করুন
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/10060 [00:00<?, ? examples/s]

Map:   0%|          | 0/2515 [00:00<?, ? examples/s]

In [27]:
# ৭. Training Arguments (সংশোধিত)
training_args = TrainingArguments(
    output_dir="./sentnob_banglabert",
    eval_strategy="epoch",          # eval_strategy ব্যবহার করুন
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

In [28]:
# ৮. Evaluation metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1': f1}
